Exercises:
- E01: Tune the hyperparameters of the training to beat my best validation loss of 2.2
- **E02: I was not careful with the intialization of the network in this video. (1) What is the loss you'd get if the predicted probabilities at initialization were perfectly uniform? What loss do we achieve? (2) Can you tune the initialization to get a starting loss that is much more similar to (1)?**
- E03: Read the Bengio et al 2003 paper (link above), implement and try any idea from the paper. Did it work?

In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [2]:
words = open('names.txt', 'r').read().splitlines()
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [7]:
# build the dataset
block_size = 3 # context length: how many characters do we take to predict the next one?

def build_dataset(words):  
  X, Y = [], []
  for w in words:

    #print(w)
    context = [0] * block_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      #print(''.join(itos[i] for i in context), '--->', itos[ix])
      context = context[1:] + [ix] # crop and append

  X = torch.tensor(X)
  Y = torch.tensor(Y)
  print(X.shape, Y.shape)
  return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

torch.Size([182580, 3]) torch.Size([182580])
torch.Size([22767, 3]) torch.Size([22767])
torch.Size([22799, 3]) torch.Size([22799])


In [40]:
g = torch.Generator().manual_seed(2147483647) # for reproducibility
C = torch.randn((27, 10), generator=g)
W1 = torch.randn((30, 200), generator=g)
b1 = torch.randn(200, generator=g)
W2 = torch.randn((200, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

In [41]:
emb = C[Xtr]
h = torch.tanh(emb.flatten(1) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Ytr)

logits_eq = torch.zeros(Xtr.shape[0], 27)
loss_eq = F.cross_entropy(logits_eq, Ytr)


print("均匀预测的理论 loss:", loss_eq.item())
print("实际初始 loss:", loss.item())

均匀预测的理论 loss: 3.295837163925171
实际初始 loss: 26.009389877319336


In [42]:
W2 *=  0.01
b2 *= 0.01

emb = C[Xtr]
h = torch.tanh(emb.flatten(1) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Ytr)

logits_eq = torch.zeros(Xtr.shape[0], 27)
loss_eq = F.cross_entropy(logits_eq, Ytr)


print("均匀预测的理论 loss:", loss_eq.item())
print("实际初始 loss:", loss.item())

均匀预测的理论 loss: 3.295837163925171
实际初始 loss: 3.306427001953125


如果只用randn，最终的概率可能会因为真实值的预测值过低导致loss很大。

因为同概率下loss相对完全随机很低，我们可以通过给原始权重乘一个很小的值来缩小概率差距，经过softmax后可以达到近似相等概率的效果。